# 27_02 라벨 생성 및 데이터 준비

In [ ]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

## 실습 — failure_soon 라벨 만들기
- RUL ≤ 30 규칙으로 failure_soon 컬럼을 직접 생성
- 27_cmapss_fd001_sample.csv 사용

### 데이터 불러오기
- 이 시점에는 failure_soon 컬럼이 아직 없는 상태


In [ ]:
# 코드

### 라벨 생성
조건식 → astype(int) → 새 컬럼에 담기

In [ ]:
# 코드

### 결과 확인
RUL과 failure_soon을 나란히 비교해 잘 붙었는지 확인

In [ ]:
# 코드

### 경계값 점검
RUL이 정확히 30인 행에서 failure_soon이 1인지 확인

In [ ]:
# 코드

## 클래스 분포 확인
- 개수와 비율을 직접 보고 불균형을 눈으로 확인
- value_counts로 0과 1의 개수와 비율을 확인
- 우리 데이터가 불균형임을 직접 눈으로 확인

### 개수 세기
0과 1이 각각 몇 개인지 출력

In [ ]:
# 코드

### 비율로 보기
normalize=True 옵션으로 개수 대신 비율 출력

In [ ]:
# 코드

## 불균형 정리
- 뒤에서 데이터를 나눌 때 stratify로 이 비율을 유지
- 1(곧 고장)의 비율이 작음 = 클래스 불균형


## 임계값 바꿔 분포 비교
- 임계값 선택이 라벨에 주는 영향을 체감하는 실험
- 임계값 20, 30, 50으로 1의 개수가 어떻게 변하는지 비교
- 임계값 선택이 라벨에 주는 영향을 체감

### 임계값 20
RUL 20 이하인 행의 개수 — 임계값 20일 때 1의 개수


In [ ]:
# 코드

### 임계값 30과 50
임계값 30과 50일 때 1의 개수를 각각 확인

In [ ]:
# 코드

## 임계값 영향 정리
- 같은 데이터인데 임계값만 바꿔도 라벨이 크게 달라짐

## 누수 체험
- 모델 없이도 간단한 비교로 누수의 위력을 확인
- RUL만으로 failure_soon을 완벽히 되살릴 수 있음을 확인


### RUL로 라벨 되살리기
모델 없이 RUL만 가지고 failure_soon을 다시 계산


In [ ]:
# 코드

### 원래 라벨과 비교
되살린 라벨과 원래 라벨이 전부 같은지 확인


In [ ]:
# 코드

## 누수의 의미
- RUL 하나로 정답이 완벽히 결정 — 입력에 넣으면 커닝
- RUL은 정답을 그대로 알려 주므로 X에서 제외
- 누수가 있으면 학습은 완벽해도 실전은 무력


## X와 y 구성하기
- 센서만 X에 넣고 RUL은 빼는 게 핵심
- 센서 6개를 X로 구성
- failure_soon을 y로 · unit_id, cycle, RUL은 제외

### feature 목록 정하기
X에 넣을 센서 6개의 이름을 목록으로 정리

In [ ]:
# 코드

### X 만들기
features 목록으로 센서 컬럼들만 골라 X에 담기

In [ ]:
# 코드

### y 만들기
정답인 failure_soon만 골라 y에 담기


In [ ]:
# 코드

### X/y 확인
입력과 정답은 같은 수의 행을 가져야 함


In [ ]:
# 코드

## 데이터 나누기
- 배운 세 가지 설정을 모두 적용해 실제로 분리
- train_test_split으로 학습용과 평가용 분리
- `test_size 0.2` · `random_state 42` · `stratify=y`

### split 도구 불러오기
scikit-learn에서 데이터를 나눠 주는 도구 불러오기

In [ ]:
# 코드

### 분리 실행
test_size · random_state · stratify 모두 지정해 네 덩어리로 분리

In [ ]:
# 코드

### 학습용 크기 확인
학습용 입력 X_train과 정답 y_train의 행 수 확인

In [ ]:
# 코드

### 평가용 크기 확인
평가용 입력 X_test와 정답 y_test의 행 수 확인

In [ ]:
# 코드

## 분리 정리
- 네 덩어리가 모두 만들어졌는지 확인
- 학습용과 평가용이 약 80 대 20으로 나뉨

## 학습/평가 클래스 비율 비교
- 두 쪽의 클래스 비율을 비교해 stratify의 효과 검증
- 학습용과 평가용의 1 비율이 비슷한지 확인


### 학습용 비율
- 학습용 정답의 0과 1 비율 확인

In [ ]:
# 코드

### 평가용 비율
평가용 정답의 0과 1 비율 확인


In [ ]:
# 코드

## stratify 효과 정리
- 학습용과 평가용 모두 원래의 불균형 비율을 그대로 유지
- 두 쪽 모두 원래 비율을 유지 = stratify 성공

## test_size 바꿔 관찰
- 비율 설정이 분할 크기에 주는 영향을 직접 관찰
- test_size를 바꾸면 학습용/평가용 크기가 달라짐

### test_size 0.3으로
test_size를 0.3으로 바꿔 다시 분할


In [ ]:
# 코드

### 크기 비교
test_size 0.2일 때와 0.3일 때의 학습용 크기를 비교


In [ ]:
# 코드